# **Strides**

# Understanding Stride in CNNs: Pros and Cons

In Convolutional Neural Networks (CNNs), the **Stride** is a hyperparameter that defines the step size—the number of pixels the filter shifts—as it slides across an input matrix.


## 1. What is Stride?

* **Stride = 1 (Default):** The filter moves one pixel at a time. It heavily overlaps with its previous positions, capturing highly detailed, local features.
* **Stride = 2:** The filter jumps two pixels at a time, skipping intermediate locations. This cuts the output dimensions roughly in half.




## 2. The Pros (Advantages of Higher Strides)

- **Natural Spatial Downsampling** : Using a stride of 2 or more compresses the width and height of the feature maps automatically. In many modern network architectures (like ResNet), this allows developers to downsample the image directly within the convolutional layer, eliminating the need for separate Max Pooling layers.

- **Reduced Computational Overhead** : Because the filter skips pixels, the total number of operations required to scan the image drops significantly. Fewer steps translate to fewer floating-point operations (FLOPs), reduced memory consumption, and significantly faster processing speeds during both training and inference.

- **Rapidly Expanding Receptive Field** : A larger stride covers the span of an input image in fewer layers. This allows deeper layers in the network to gain a broad, global perspective of the entire scene much faster, rather than staying focused on small, localized pixel groupings.
- **Use it only when you want high level features from the img, and want to avoid the low level features (details)**


## 3. The Cons (Disadvantages of Higher Strides)

- **Drastic Loss of Fine-Grained Information** : When a filter skips over pixel coordinates, it misses the fine structural data contained within those skipped regions. Small, critical features such as intricate textures, thin lines, or minor color variations can easily slip through the gaps.

- **Poor Localization Precision** : Aggressive striding compromises spatial fidelity. Because the output grid is deeply compressed, the exact pixel coordinates of an extracted feature are blurred. This makes high strides a poor choice for tasks that require absolute pixel-perfect precision, such as **Object Detection** (drawing bounding boxes) or **Semantic Segmentation** (mapping exact outlines).

- **Overly Abstract Representations Early On** : Forcing a network to downsample too aggressively in the very first layers can degrade overall accuracy. If a model compresses data before learning basic foundational features (like corners or edges), it struggles to build robust, higher-level representations later on.


## Summary Matrix

| Stride Selection | Output Map Size | Computational Load | Feature Retention | Ideal Placement |
| :--- | :--- | :--- | :--- | :--- |
| **Low Stride ($S = 1$)** | Large / Preserved | High (Slower) | High Detail & Maximum Overlap | Early layers to extract fine details, edges, and raw textures. |
| **High Stride ($S \ge 2$)** | Small / Compressed | Low (Faster) | Coarse / Abstract Components | Deeper layers where global context and speed matter more than pixel locations. |


## **IMP**
If you have an img of 5 x 5, and have used stride and padding, then after using kernel the output img will be : <br>
- With Stride only
    - (((n-f)/s) + 1) x (((n-f)/s) + 1)
    - If you sometimes get float values from '((n+2p-f)/s)', then you have to apply floor to the number.
        - If the float is 1.1 or 1.9, it will be counted as 1
- With Stride and padding
    - (((n+2p-f)/s) + 1) x (((n+2p-f)/s) + 1)
    - If you sometimes get float values from '((n+2p-f)/s)', then you have to apply floor to the number.
        - If the float is 1.1 or 1.9, it will be counted as 1


Here s represents Stride values.


---

**If i have a 5 x 5 img and i have applied padding and it is now giving me 7 x 7 img, after using the filter the img that i get is same 5 x 5. But my doubt is, if i increase the padding to such an extent so i can apply stride, and after doing the filter thing the output img that i get is still 5 x 5, then would there be any meaning to applying stride in that case ???**

When you apply a high amount of padding to counteract a large stride and force the output dimensions to remain identical to the input dimensions, the mathematical pipeline still functions, but the physical nature of the features being extracted completely shifts.


## 1. The Mathematical Setup

Consider a $5 \times 5$ input ($W=5$) and a standard $3 \times 3$ filter ($K=3$). 

* **Standard Approach (Stride = 1):** Requires **Padding = 1** to get a $5 \times 5$ output. The image expands slightly to $7 \times 7$.
* **Your Scenario (Stride = 2):** To force a $5 \times 5$ output, you must scale up to **Padding = 3**. This expands the original $5 \times 5$ image into a massive $11 \times 11$ matrix entirely surrounded by zeros before filtering begins.




## 2. The Pros (Why Someone Would Do This)

 - **Strict Structural Grid Consistency** : In complex multi-branch network architectures (like U-Net or Residual Blocks), tensors from different layers must be added or concatenated together. Keeping the output size precisely at $5 \times 5$ using stride and padding allows you to alter the receptive field without breaking the spatial compatibility required for these skips and shortcuts.

 - **High Border and Edge Sensitivity** : Because the filter spends a massive portion of its steps transitioning from the heavy zero-padded fields into the actual real pixels, the network becomes incredibly sensitive to boundaries. It learns exactly *how* and *where* features hit the outermost edges of your frame.

 - **Fast Receptive Field Growth with No Resolution Loss** : Usually, if you want a layer to look at a broader context using a stride of 2, you have to sacrifice image resolution. By offsetting it with high padding, you get the benefit of a stride (the filter jumps farther across the image space to see global relationships) while preserving your core resolution.


## 3. The Cons (The Heavy Drawbacks)

 - **Heavy Computation on "Ghost" Data (Zeros)** : In an $11 \times 11$ expanded matrix, there are 121 total pixels, but only 25 of them contain real image datThe remaining 96 pixels are purely padded zeros. Your GPU spends a massive amount of floating-point operations multiplying weights by zero, resulting in heavily wasted computational cycles.

 - **Artificial Border Artifacts (The Edge Effect)** : Since the outermost pixels of your final $5 \times 5$ output map are computed almost entirely over empty zero fields, they do not hold genuine visual features. This can introduce "dead zones" around the perimeter of your feature maps, which often degrades the classification accuracy of centralized objects.

 - **Dilution of Internal Textures** : Because the filter is stepping by 2 pixels over a heavily padded space, it takes fewer total looks at the inner, core pixels of your actual image. The subtle internal details, textures, and patterns are heavily diluted in favor of global positioning.


## Summary Matrix

| Metric | Standard "Same" ($S=1, P=1$) | Extreme Offset ($S=2, P=3$) | Impact on Model |
| :--- | :--- | :--- | :--- |
| **Output Size** | $5 \times 5$ | $5 \times 5$ | Dimension matching is maintained across both configurations. |
| **Padded Matrix Size** | $7 \times 7$ (49 pixels) | $11 \times 11$ (121 pixels) | The extreme offset forces the model to process 2.5x more background space. |
| **Information Focus** | Concentrated on local internal features. | Concentrated on boundary entry/exit points. | Completely changes the layer's objective from feature extraction to border tracking. |
| **Computational Efficiency** | Optimized. | Highly wasteful (mostly calculating zeros). | Fine for specialized architectures, but highly inefficient for standard classification tasks. |